# 04 - Governed Text-to-SQL Agent

A deterministic text-to-SQL planner whose EVERY draft is validated before it is
returned: `pass` ships, `warn` is revised to a minimized aggregate, `fail` is
refused with the policy reason.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
SAFE_SQL = "SELECT region, SUM(arr) FROM customers GROUP BY region"

def plan(question):
    q = question.lower()
    if "marketing" in q or "campaign" in q:
        return (
            "SELECT customer_name, email FROM customers WHERE marketing_consent = 'opted_in'",
            "purpose.prohibited_use",
        )
    if "email" in q or "identify" in q:
        return (
            "SELECT customer_name, email FROM customers WHERE region = 'EU'",
            "purpose.allowed_use",
        )
    return (SAFE_SQL, "purpose.allowed_use")

def text_to_sql(question):
    sql, scenario_key = plan(question)
    answer = client.validate_query_context(
        sql,
        scenario_key=scenario_key,
        default_database="acmecloud_demo",
        default_schema="public",
    )
    verdict = answer["verdict"]
    if verdict == "fail":
        return {"question": question, "verdict": verdict, "sql": None}
    if verdict == "warn":
        return {"question": question, "verdict": verdict, "sql": SAFE_SQL}
    return {"question": question, "verdict": verdict, "sql": sql}


In [ ]:
for question in [
    "How does ARR break down by region?",
    "List EU customers with their email addresses.",
    "Build an email list for the marketing campaign.",
]:
    result = text_to_sql(question)
    print(f"{result['question']}")
    print(f"  verdict: {result['verdict']}  sql: {result['sql']}")
